In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# CONFIG

In [2]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 10
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [6]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [7]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [8]:
print(df.head())
print(df.columns.tolist())

   row_id                                               body  \
0       0  Banks don't want you to know this! Click here ...   
1       1  SD Stream [ ENG Link 1] (http://www.sportsstre...   
2       2  Lol. Try appealing the ban and say you won't d...   
3       3  she will come your home open her legs with  an...   
4       4  code free tyrande --->>> [Imgur](http://i.imgu...   

                                                rule      subreddit  \
0  No Advertising: Spam, referral links, unsolici...     Futurology   
1  No Advertising: Spam, referral links, unsolici...  soccerstreams   
2  No legal advice: Do not offer or request legal...   pcmasterrace   
3  No Advertising: Spam, referral links, unsolici...            sex   
4  No Advertising: Spam, referral links, unsolici...    hearthstone   

                                  positive_example_1  \
0  If you could tell your younger self something ...   
1  [I wanna kiss you all over! Stunning!](http://...   
2  Don't break up wi

In [9]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        tmp= list(set(tmp))#deduplication
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [10]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len,df=None,df_test=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        if df is not None:
            extras=add_data(df)
            self.texts+= extras[0]
            self.labels+= extras[1]
            print('Added additional data of size from train examples',len(extras[1]))
        if df_test is not None:
            extras=add_data(df_test)
            self.texts+= extras[0]
            self.labels+= extras[1]
            print('Added additional data of size from test examples',len(extras[1]))

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [11]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.2)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [12]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [13]:
def validate(model, loader):
    model.eval()
    preds, targets = [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
    auc = roc_auc_score(targets, preds)
    val_loss = total_loss / len(loader)
    return auc, val_loss ,np.array(preds)

In [14]:
# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
all_preds = []
folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
for fold, (tr_idx, val_idx) in enumerate(folds.split(df, df["label"])):
    print(f"\n===== Fold {fold + 1} =====")
    df_test = pd.read_csv(test_path)
    # Get train texts as a set for faster lookup
    train_ds = JigsawDataset(
        df.iloc[tr_idx]['text'].tolist(), 
        df.iloc[tr_idx]['label'].tolist(), 
        tokenizer, MAX_LEN, df=df,df_test=df_test[df_test['rule'].str.lower().isin(df['rule'].str.lower().drop_duplicates())]
    )
    
    # Get all training texts (original + additional data)
    all_train_texts = set(train_ds.texts)
    
    # Filter validation indices to exclude any text present in the complete training set
           # AND ensure uniqueness in validation set
    val_texts = df.iloc[val_idx]['text'].tolist()
    seen_val_texts = set()
    non_overlapping_unique_val_idx = []
    
    for i, val_text in enumerate(val_texts):
        # Check if text is not in training set AND not already seen in validation
        if val_text not in all_train_texts and val_text not in seen_val_texts:
            non_overlapping_unique_val_idx.append(val_idx[i])
            seen_val_texts.add(val_text)
    
    selected_idx = np.array(non_overlapping_unique_val_idx)
    print(f'\nLength {len(val_idx)} -> {len(selected_idx)}\n')
    # Create validation dataset with filtered indices
    val_ds = JigsawDataset(
        df.iloc[selected_idx]['text'].tolist(), 
        df.iloc[selected_idx]['label'].tolist(), 
        tokenizer, MAX_LEN
    )

    train_ds = JigsawDataset(
        df.iloc[tr_idx]['text'].tolist(), 
        df.iloc[tr_idx]['label'].tolist(), 
        tokenizer, MAX_LEN, df=df, df_test=df_test
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    # Initialize classification model and load MLM pre-trained weights
    model = JigsawModel(MODEL_PATH).to(DEVICE)
    
    for name, param in model.named_parameters():# maek few llayers trainable, not all layers, val degudding  to correct data 
        if name.startswith('base.'):# and not ('10' in name or '11' in name):
            param.requires_grad = False
            # print(f"Set {name} to non-trainable")
    print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
    # break

    # model = torch.nn.DataParallel(model)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    best_loss = 10
    best_auc=0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, None)
        val_auc, val_loss ,val_preds = validate(model, val_loader)
        
        if epoch<=6:
            for name, param in model.named_parameters(): 
                if name.startswith('base.') and (str(11-epoch) in name):
                    param.requires_grad = True
                    
        print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), f"model_fold{fold}.bin")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")

    all_preds.append(pd.Series(val_preds))


===== Fold 1 =====
Added additional data of size from train examples 1907
Added additional data of size from test examples 38

Length 406 -> 172

Added additional data of size from train examples 1907
Added additional data of size from test examples 38


2025-09-20 18:22:01.695515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758392522.054809      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758392522.164479      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Trainable Params:  769
Epoch 1/10


100%|██████████| 112/112 [00:33<00:00,  3.38it/s]


Loss: 0.7033, Val Loss: 0.6967, Val AUC: 0.4507
Epoch 2/10


...

Loss: 0.0833, Val Loss: 1.2558, Val AUC: 0.7028
Epoch 9/10


100%|██████████| 112/112 [01:16<00:00,  1.47it/s]


Loss: 0.0643, Val Loss: 1.2982, Val AUC: 0.7006
Epoch 10/10


100%|██████████| 112/112 [01:16<00:00,  1.46it/s]


Loss: 0.0679, Val Loss: 1.4475, Val AUC: 0.6893


In [15]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [16]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv